# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanay1122/flyrank-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)


## 1. Question

**Research question:** for a portfolio that loses measurable search traffic every month, can a trained model rank the pages a *human* should review first — better than the existing hand rule — using only information available at decision time?

**The decision it supports:** a reviewer's *reading order* for the coming week. Nothing more, nothing less — no auto-actions, no promises about the search engine. This is a decision-support ranking problem, stated up front, and every later section stays inside that box.

**Why it matters in one scenario:** in the studied slice, 3-page-in-20 outcomes declined by >30% relative to their first-3-weeks pace within 11 days after the observation window closed. The hand rule catches a third of those in its top-20 reading list. A model that doubles that hit rate changes which pages are opened *this week* — while the blind-spot work (section 5) keeps the model out of the "coverage" business.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Everything the question needs, from the committed receipts (no warehouse call needed here).
import json

from pathlib import Path
def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("run from inside the repo")

from pathlib import Path

REPO = find_repo_root()

w04_sig = json.load(open(REPO / "work" / "outputs" / "w04_signal_receipt.json", encoding="utf-8"))
w04_base = json.load(open(REPO / "work" / "outputs" / "w04_baseline_receipt.json", encoding="utf-8"))
w05 = json.load(open(REPO / "work" / "outputs" / "w05_model_receipt.json", encoding="utf-8"))
w06 = json.load(open(REPO / "work" / "outputs" / "w06_validation_receipt.json", encoding="utf-8"))
w07 = json.load(open(REPO / "work" / "outputs" / "w07_playbook_receipt.json", encoding="utf-8"))
print("receipts loaded:", "w04_signal, w04_baseline, w05_model, w06_validation, w07_playbook")
print(f"question anchors: {w05['frame_pages']:,} pages | base rate {w05['base_rate_pct']}% decline_11d | rule P@20 {w04_base['precision_at_20']} | model P@20 {w05['comparison_table'][1]['P@20']}")
print("decision supported: a reviewer's reading order (ranked queue + rails), never an automated action")

receipts loaded: w04_signal, w04_baseline, w05_model, w06_validation, w07_playbook
question anchors: 83,258 pages | base rate 15.9% decline_11d | rule P@20 0.35 | model P@20 0.55
decision supported: a reviewer's reading order (ranked queue + rails), never an automated action


## 2. Data

**Source:** the public-safe warehouse release hosted on Hugging Face as `FlyRank/internship-warehouse` (gated, read-only access; ~78.8M rows of daily performance across 104 pseudonymous clients). This lane works on one mid-panel month — `month=2026-03` (the panel's final month remains a sealed evaluation lane by design and was never used to build features).

**Grain and windows:** one row per pseudonymous page per day. Features aggregate `report_date <= 2026-03-20` with GSC data available; the label aggregates **03-21..31** and never appears on the left side of any feature. `dim_content` contributes decision-time metadata (word count, backlinks, intent, type, created/updated dates).

**Exclusions, disclosed:** pages below the quality floor (SUM impressions < 100 or COUNT < 10 days) never enter; then **314** pages absent from the label window and **37** pages with zero early-week pace are dropped — a selection choice (they are unknowable this month), stated here rather than hidden. Every drift check starts with these numbers.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# The data receipts, printed from the committed frame receipts (identical across w04->w07).
print("package: fact_content_daily_performance (month=2026-03), dim_content, dim_clients | release build v20260703")
print(f"frame: {w05['frame_pages']:,} pages (floor SUM imps>=100, COUNT>=10) | drops disclosed: 314 absent from label window, 37 no-pace")
print(f"feature window: report_date <= 2026-03-20 (gsc_data_available IS TRUE) | label window: 2026-03-21..31")
print(f"clients: {w05['split']['clients']} | pseudonymous hashes only (client_hash_id / content_hash_id)")
w04_counts = w04_base.get("action_counts", {})
print(f"hand-rule queue scale for context: {w04_counts}")

package: fact_content_daily_performance (month=2026-03), dim_content, dim_clients | release build v20260703
frame: 83,258 pages (floor SUM imps>=100, COUNT>=10) | drops disclosed: 314 absent from label window, 37 no-pace
feature window: report_date <= 2026-03-20 (gsc_data_available IS TRUE) | label window: 2026-03-21..31
clients: 36 | pseudonymous hashes only (client_hash_id / content_hash_id)
hand-rule queue scale for context: {'no_action': 65032, 'watch': 18092, 'refresh': 134}


## 3. Methodology

**Features (21, all strictly decision-time):** momentum capped at 10x, log impressions, log click rate, position mean, row staleness, days active, metadata staleness + a `meta_unknown` flag (neutralizes post-decision `content_updated_date` — 80.6% of pages cannot resolve an update age at decision time), a band-relative CTR-fix flag, log search volume, competition, log word count, log backlinks, and intent×type dummies. No label-window columns anywhere — asserted by name at audit time.

**Label** (`decline_11d`): the page's mean target-window impressions (03-21..31) fell below **0.70×** its mid-month pace — a relative-to-self threshold, not an absolute one.

**Baseline:** the Week-4 hand rule (staleness + pace + impact points), AUC 0.544 / P@20 0.35 (single-axis receipt).

**Validation:** client-grouped 5-fold GroupKFold — every test page scored by a model that never saw its client; 36 clients, all folds disjoint (asserted per run, seeds fixed at 0). The random-split row is re-printed beside the honest rows everywhere as the memorization exhibit, never cited as skill.

**Leakage checks** (the Week-6 audit, run on the final feature set): strict timeline assertion, label-window name scan, the **confession test** (deliberately adding the label's own input pushed grouped OOF AUC 0.659 → 1.000, then removed — the harness demonstrably catches leaks), product-flag scan, population disclosure, and top-feature sanity (permutation top lever = `momentum_capped`, matching the audited risk-UP direction).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Methodology receipts: validation design + the leakage audit, printed from the committed w05/w06 receipts.
print("features: 21 (momentum capped 10x, log imps/CTR/wordcount/backlinks, pos, staleness + meta_unknown neutralization, ctr_fix_flag, search/competition, intent x type dummies)")
print("label: decline_11d = target-window mean < 0.70 x mid-month pace (03-21..31); features strictly <= 03-20")
print(f"baseline: hand rule AUC {w04_base['rule_auc']} / P@20 {w04_base['precision_at_20']} (w04 receipt)")
print(f"split: {w05['split']['design']} | {w05['split']['clients']} clients | all_folds_client_disjoint = {w05['split']['all_folds_client_disjoint']} | fixed seeds 0")
la = w06["leak_audit"]["checklist"]
print("leakage audit (w06, final feature set), item by item:")
for k, v in la.items():
    print(f"  [{'x' if v['ok'] else ' '}] {k}")
print(f"confession test: deliberately-mined leakage drove grouped OOF AUC {w06['leak_audit']['honest_oof_auc']} -> {w06['leak_audit']['confession_test_auc']}, then removed")

features: 21 (momentum capped 10x, log imps/CTR/wordcount/backlinks, pos, staleness + meta_unknown neutralization, ctr_fix_flag, search/competition, intent x type dummies)
label: decline_11d = target-window mean < 0.70 x mid-month pace (03-21..31); features strictly <= 03-20
baseline: hand rule AUC 0.544 / P@20 0.35 (w04 receipt)
split: client-grouped 5-fold (GroupKFold) | 36 clients | all_folds_client_disjoint = True | fixed seeds 0
leakage audit (w06, final feature set), item by item:
  [x] timeline_strict
  [x] meta_date_neutralized
  [x] no_label_siblings
  [x] confession_test
  [x] no_product_flags
  [x] population_disclosed
  [x] grouped_split
  [x] base_rate
  [x] top_feature_sane
  [x] oof_metrics
confession test: deliberately-mined leakage drove grouped OOF AUC 0.659 -> 1.0, then removed


## 4. Results (vs baseline)

Same data, same split, same label — base rate **15.9%** printed next to every number at every K.

| split | AUC | P@20 | P@100 | P@500 |
|---|---|---|---|---|
| Baseline hand rule | 0.544 | 0.35 | 0.28 | 0.32 |
| **LR (this work)** | **0.659** | **0.55** | **0.47** | **0.48** |
| RF complexity check | 0.655 | 0.25 | 0.38 | 0.45 |
| random-split exhibit | 0.674 | 0.85 | 0.68 | 0.52 |

**How to read it:** the queue-relevant metric is P@20 — the share of a 20-page reading list that actually declined. The model holds **0.55 vs 0.35** for the rule (3.2× the random pick rate of 15.9%); at 100 and 500 pages the multiplicative gain narrows but the ordering advantage persists. The RF matched the LR's fold-mean AUC (0.657 vs 0.656) without any top-of-queue gain — complexity did not earn its keep here. The random-split row is printed **as the exhibit it is**: it wobbles between reruns (P@20 across reruns: 0.80–0.90), while the grouped rows reproduce to the digit — the second memorization receipt after the 1.5-pt AUC gap itself.

A stability subtlety this work surfaced (and keeps): the *shuffled* split fixes random positions over a frame whose row order comes out of a parallel GROUP BY — so reruns wobble only that row, while the grouped protocol is order-invariant. Honest takeaway: cite the grouped numbers or nothing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# The honest table, printed straight from the committed receipts (multiple receipts must agree).
rows = w05["comparison_table"]
print("same data, same split family per row, base rate 15.9% beside every K:")
for r in rows:
    print(f"  {r['design']:<62} AUC {r['AUC']:>5.1f} | P@20 {r['P@20']:.2f} | P@100 {r['P@100']:.2f} | P@500 {r['P@500']:.2f}")
print(f"fold AUC: LR {w05['fold_auc']['lr']} | RF {w05['fold_auc']['rf']}")
print(f"memorization exhibit (shuffled): AUC {next(r for r in rows if 'shuffled' in r['design'])['AUC']} | "
      f"P@20 ranges 0.80-0.90 across three reruns; grouped rows reproduce exactly (w06 stability note)")
print(f"queue-level read (w07): model top-40 held 23 decliners ({w07['queue']['model_top40']['decline_pct_observed']}%) "
      f"vs ~6 expected at the {w05['base_rate_pct']}% base rate; zero SIZE_OF_PRIZE pages made the top 40 -> the size rail exists")

same data, same split family per row, base rate 15.9% beside every K:
  Baseline: Week-4 hand rule (ties -> volume, w04 queue order)   AUC  54.4 | P@20 0.35 | P@100 0.28 | P@500 0.32
  Logistic Regression, client-grouped OOF                        AUC  65.9 | P@20 0.55 | P@100 0.47 | P@500 0.48
  Random Forest, client-grouped OOF                              AUC  65.5 | P@20 0.25 | P@100 0.38 | P@500 0.45
  LogReg shuffled split - DISHONEST receipt, not a claim         AUC  67.4 | P@20 0.85 | P@100 0.68 | P@500 0.52
fold AUC: LR [0.67, 0.645, 0.686, 0.634, 0.647] | RF [0.68, 0.672, 0.673, 0.629, 0.633]
memorization exhibit (shuffled): AUC 67.4 | P@20 ranges 0.80-0.90 across three reruns; grouped rows reproduce exactly (w06 stability note)
queue-level read (w07): model top-40 held 23 decliners (57.5%) vs ~6 expected at the 15.9% base rate; zero SIZE_OF_PRIZE pages made the top 40 -> the size rail exists


## 5. Limitations

What this work cannot claim — written by the author, not by a reviewer:

- **No causal claim anywhere.** No experiment ran; every number is an association observed on one slice. "Momentum spikes decline more often" is a measured, directional statement about 11-day outcomes in one month of one portfolio — not "refreshing or slowing momentum will change outcomes."
- **One slice.** Month 2026-03, one 11-day label window, one population (83,258 pages with its disclosed 314 + 37 drops). Other months need the grouped protocol re-run before any reuse.
- **The blind spot is structural, not a bug:** 13,037 of 13,277 decliners (98%) rank beyond 500. The queue is review-order, never coverage — the size-of-prize rail exists precisely because score and prize are different axes.
- **Metadata opacity:** 80.6% of pages have unresolvable update ages at decision time; the staleness verdict stays MIXED, so staleness is a lookup code, not a risk lever.
- **Reproducibility wobble on exhibit rows:** random-split receipts wobble between reruns (row-order sensitivity); grouped receipts do not. Nothing in the published claims rests on a shuffled number.
- **Pseudonymous data:** ids are hashes; the human checks behind META_UNKNOWN/CTR_LEAK require real CMS access this dataset does not include.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Exactly the numbers that would tempt over-claiming, printed with the bound each one carries.
print(f"blind spot: {w07['blind_spot_rule']}")
print(f"meta unknown: {w05['meta_unknown_pages']:,} of {w05['frame_pages']:,} pages ({100 * w05['meta_unknown_pages'] / w05['frame_pages']:.1f}%) -> verdict {w04_base['verdictA2_staleness_metadata']}")
print(f"selection: {w07['disclosed_drops']} (absent = unknowable this month; the playbook says nothing about them)")
print("no-go summary from w07 receipt:")
for item in w07["no_go_list"]:
    print("  x", item)

blind spot: 13,037 of 13,277 decliners (98%) rank beyond 500 (w05 receipt); queue is review-order, not coverage
meta unknown: 67,109 of 83,258 pages (80.6%) -> verdict MIXED
selection: {'absent_label_window': 314, 'no_pace': 37} (absent = unknowable this month; the playbook says nothing about them)
no-go summary from w07 receipt:
  x no auto-refresh on a staleness timer (verdict MIXED; unknown-metadata share 80.6%)
  x no auto-publish/archive/noindex on score alone (P@20 0.55 at base 15.9%)
  x no causal wording (no experiment ran; claim ladder ends at decision-support)
  x no skip-page rules from ranks (98% of decliners live beyond rank 500)
  x no quoting of the random-split row to humans (wobble exhibit 0.80-0.90 P@20)


## 6. Ranked recommendations

The playbook output (Week 7), as the paper's recommendations section — decision-support only:

1. **Work the two rails, in this order.** Model rail first (top 40 by grouped out-of-fold score — this slice held 23 decliners among 40, ~4× the base rate), then the size-of-prize rail (the 10 biggest pages outside the top 40; in this slice they carried **1,416,001** early-window impressions combined and every one was also META_UNKNOWN — the model's top 40 contained zero of them).
2. **Read reason codes as lookup orders, not verdicts.** `MOMENTUM_SPIKE` (29.8% decline, n=5,763 vs 15.9% base — check the pace story for a real movement), `CTR_LEAK` (below-band-median click rate; w04's observed low-vs-high gaps of +11.0/+7.7/+7.3 pp in the top-3/4-10/11-30 bands, n≥30 each — decide title vs snippet vs nothing), `META_UNKNOWN` (resolve first, always), `SIZE_OF_PRIZE` (worth a look regardless of score).
3. **Keep the human in the loop.** The 5-item no-go list (no auto-refresh on staleness timers; no publish/archive on score alone; no causal wording; no skip-page rules from rank; no quoting shuffled rows) is part of the deliverable, not a footnote.
4. **Monitor against the committed thresholds.** Six triggers, each tied to a receipt (grouped AUC < 0.60, P@20 < 0.40, base rate outside 13–19%, meta-unknown share shift, client-set change, new month partition → full rebuild).
5. **Before scaling: the next experiment.** The refresh play (the paper's #1 suggestion, popular in the domain) remains unproven on this lane's 11-day label (staleness verdict MIXED) — the honest next step is a designed, age-matched refresh experiment, not a bigger observational model.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Receipt for every recommendation sentence (numbers only from the w07 receipt).
print("code table (top-40 presence):", w07["queue"]["code_counts_top40"])
print("rail:", w07["queue"]["size_rail"])
print("archetype decline rates beside base rate:")
for r in w07["reason_codes"]["table"]:
    print(f"  {r['code']:<15} n={r['n']:>6,} decline {r['decline_pct']}% (share {r['share_pct']}%) vs base {w05['base_rate_pct']}%")
print("triggers:", len(w07["monitor_triggers"]), "| no-go:", len(w07["no_go_list"]))

code table (top-40 presence): {'MOMENTUM_SPIKE': 37, 'CTR_LEAK': 35, 'META_UNKNOWN': 29}
rail: {'n': 10, 'imps_sum': 1416001}
archetype decline rates beside base rate:
  MOMENTUM_SPIKE  n= 5,763 decline 29.8% (share 6.9%) vs base 15.9%
  CTR_LEAK        n=37,137 decline 19.4% (share 44.6%) vs base 15.9%
  SIZE_OF_PRIZE   n= 8,328 decline 7.2% (share 10.0%) vs base 15.9%
  META_UNKNOWN    n=67,109 decline 15.1% (share 80.6%) vs base 15.9%
triggers: 6 | no-go: 5


## 7. Artifacts the paper embeds

The deployed page embeds three artifacts, generated here into `docs/img/` (the repo IS the website via GitHub Pages, `/docs` folder):

- `fig_results_precision_at_k.png` — the one-chart summary: review-first precision at 20, all rows on the same evaluation, base rate drawn on the plot.
- `fig_w07_reason_codes_top40.png` — which reason codes actually populate the review queue.
- `fig_w07_code_decline_vs_base.png` — decline rate per reason code vs the base rate.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Generates the paper's chart trio from committed receipts ONLY (fast, no warehouse call).
import shutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

docs_img = REPO / "docs" / "img"
docs_img.mkdir(parents=True, exist_ok=True)

# (a) the results chart: P@20 rows on one honest axis
vals = [
    ("base rate (random)", w05["base_rate_pct"]),
    ("hand rule", 100 * w04_base["precision_at_20"]),
    ("LR model (this work)", 100 * next(r for r in w05["comparison_table"] if r["design"].startswith("Logistic Regression"))["P@20"]),
    ("RF complexity check", 100 * next(r for r in w05["comparison_table"] if r["design"].startswith("Random Forest"))["P@20"]),
]
shuffled_row = next(r for r in w05["comparison_table"] if "shuffled" in r["design"])
labels = [v[0] for v in vals] + ["random-split exhibit\n(not a claim)"]
heights = [v[1] for v in vals] + [100 * shuffled_row["P@20"]]
colors = ["#9aa7b5", "#5878a4", "#2f5d8f", "#5878a4", "#c9b18a"]
fig, ax = plt.subplots(figsize=(7.2, 3.9))
bars = ax.bar(labels, heights, color=colors)
bars[-1].set_hatch("//")
ax.axhline(w05["base_rate_pct"], color="#a4453e", linestyle="--", label=f"base rate {w05['base_rate_pct']}%")
ax.set_ylabel("P@20 (%)")
ax.set_title("Review-first precision at 20 (same data, grouped split; random-split row is the exhibit)")
ax.grid(axis="y", alpha=0.3)
ax.legend()
for b, h in zip(bars, heights):
    ax.text(b.get_x() + b.get_width() / 2, h + 1.5, f"{h:.0f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(docs_img / "fig_results_precision_at_k.png", dpi=110)
plt.close(fig)

# (b)+(c) the playbook charts from w07, copied into the page's image dir
for name in ("fig_w07_reason_codes_top40.png", "fig_w07_code_decline_vs_base.png"):
    src = REPO / "work" / "outputs" / "figures" / name
    if src.exists():
        shutil.copy(src, docs_img / name)
print("paper artifacts in docs/img/:", sorted(p.name + f" ({p.stat().st_size // 1024} KB)" for p in docs_img.glob("*.png")))

paper artifacts in docs/img/: ['fig_results_precision_at_k.png (38 KB)', 'fig_w07_code_decline_vs_base.png (22 KB)', 'fig_w07_reason_codes_top40.png (18 KB)']


## 8. Demo outline (ML-12) — the 5-minute story, the social cut, the employer 3-sentencer

**Five-minute demo (question → method → one chart → one honest result → one recommendation):**

1. **Question (30s):** "In a portfolio that sheds search traffic every month, which pages does a human open first — and can a model rank that order better than our hand rule?"
2. **Method (60s):** 21 features, all readable at decision time; label = relative-to-self decline in the next 11 days; client-grouped split (the model never sees a client it trains on); the confession test proves the harness would catch a leak.
3. **One chart (45s):** P@20 — hand rule 0.35, model 0.55, base rate 0.159, and the random-split exhibit at 0.85 shown hatched as *not a claim*.
4. **Honest result (60s):** the model doubles the top-of-list hit rate (0.55 vs 0.35) but sees only 2% of the label window's decliners past rank 500 — so the playbook ships with a second rail for the big pages and a no-go list.
5. **Recommendation (45s):** use the queue to order human review, keep the no-go list on the wall, monitor the six triggers, and run the designed refresh experiment before scaling.

**Social post cut:**

> 16% of a content portfolio declined in just 11 days. A 21-feature model, grouped by client, ranked the risk with P@20 0.55 vs 0.35 for the hand rule (base rate 15.9%) — but it alone is not enough: 98% of decliners sat beyond rank 500, so the playbook pairs the queue with a size-of-prize rail and a written no-go list. Receipts: https://github.com/tanay1122/flyrank-ml · Paper: https://tanay1122.github.io/flyrank-ml/

**Employer 3-sentencer:**

> I built and validated a decline-risk ranking model over a 79M-row search-rights warehouse (83,258 pseudonymous pages, 36 clients, label = relative decline in the following 11 days), beating the existing hand rule at review-first precision (P@20 0.55 vs 0.35, AUC 0.659 vs 0.544, every fold client-disjoint). I turned the score into a review-first playbook — reason codes, a size-of-prize rail for the pages the model walks past, a five-item no-go list and six monitoring triggers — every number tied to a committed receipt. The work is deployed as a research page and fully reproducible from the repo (fixed seeds, grouped splits, honest-limitations sections written before any reader could write them).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# The demo line that must stay true when spoken out loud:
print("demo anchor chart: docs/img/fig_results_precision_at_k.png")
print(f"spoken numbers: P@20 {0.55} vs {0.35} (rule) vs {w05['base_rate_pct'] / 100:.3f} (base) | blind spot {w07['blind_spot_rule']}")
print("demo rule: one chart, one honest sentence, one recommendation - no causal wording anywhere")

demo anchor chart: docs/img/fig_results_precision_at_k.png
spoken numbers: P@20 0.55 vs 0.35 (rule) vs 0.159 (base) | blind spot 13,037 of 13,277 decliners (98%) rank beyond 500 (w05 receipt); queue is review-order, not coverage
demo rule: one chart, one honest sentence, one recommendation - no causal wording anywhere


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.